# TAB 68: exp68_unified_opf (EXP-68: Unified OPF-JEPA with Dodd-Deeds Spectral Conditioning & Normalized L1) - Colab
# Triết lý thiết kế EXP-68 (Bẻ gãy nghịch lý c_B và giải quyết triệt để sụp đổ động lực học):
# 1. SINGLE UNIFIED LATENT 64D (--unified_latent true, --latent_disentanglement false): Loại bỏ hoàn toàn sự chia tách 2-subspace [z_c, z_s] cồng kềnh với 8 loss phụ đánh nhau; quay về 1 latent thống nhất cho cả đại diện và downstream.
# 2. DODD-DEEDS SPECTRAL SIGNATURE (--spectral_condition_token true): Trích xuất c_B thuần túy từ pha Fourier (theta_k) và tỷ lệ phân bố năng lượng phổ chuẩn hóa (E_k / sum(E)), triệt tiêu 100% biên độ DC tĩnh và bất biến tuyệt đối với lift-off.
# 3. ADALN PREDICTOR MODULATION (--condition_modulation adaln): Đưa c_B vào Predictor qua Adaptive LayerNorm khởi tạo near-zero, giữ vai trò identity ban đầu, ép Cross-Attention lấy 100% hình thái khuyết tật từ Context A.
# 4. SCALE-INVARIANT NORMALIZED L1 (--pred_loss_type normalized_l1): Chuẩn hóa probe-wise trong không gian đặc trưng, triệt tiêu hoàn toàn vòng xoáy tử thần (Death Spiral) làm co rút biên độ về 0.
# 5. FULL-RANK VICREG 64D (--var_weight 0.5, --cov_weight 0.2, --anti_hijack_ratio 0.0): Loại bỏ trần thắt cổ Anti-Hijack Guard, duy trì rank biểu diễn bền vững > 20D.
# 6. SCHEDULED SCAN-CENTERED INVARIANCE (--inv_weight 0.2, --inv_start_epoch 11, --inv_ramp_epochs 20, --scan_centered_inv true): Rèn luyện tính bất biến sau warmup mà không làm sập rank.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

In [ ]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git pull --ff-only && git log --oneline -3

In [ ]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"


In [ ]:
# Symlink runs sang Drive: checkpoint va metrics ghi truc tiep len Google Drive
!mkdir -p "$DRIVE_RUNS"
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ===== EXP-68 CONFIG: Unified OPF-JEPA (Dodd-Deeds Spectral Signature + Normalized L1) =====
EXP_NAME = 'exp68_unified_opf'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement false --unified_latent true --two_stream_backbone false --embed_dim 64 --var_weight 0.5 --cov_weight 0.2 --var_gamma 1.0 --scan_centered_vicreg true --inv_start_epoch 11 --inv_ramp_epochs 20 --inv_cosine true --scan_centered_inv true --inv_weight 0.2 --diverge_patience 0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --spectral_condition_token true --use_condition_token true --condition_modulation adaln --pred_loss_type normalized_l1 --anti_hijack_ratio 0.0 --self_focus_gamma 0.0'

os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print('EXPERIMENT:', EXP_NAME)
print('FLAGS:', BASE_FLAGS + ' ' + EXTRA)


In [ ]:
# Cell Train (Fresh Run): Huấn luyện EXP-68 Unified OPF-JEPA
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

In [ ]:
# Cell Resume Train (Neu bi ngat ket noi Colab):
# Tu dong tim checkpoint moi nhat cua exp68 trong Drive va tiep tuc training
import glob, os
ckpts = sorted(glob.glob(f"{DRIVE_RUNS}/{EXP_NAME}_*/checkpoints/latest_checkpoint.pt"))
if ckpts:
    latest_ckpt = ckpts[-1]
    print(f"Resuming from: {latest_ckpt}")
    !cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --resume_checkpoint "{latest_ckpt}" --data_dir "$DATA"
else:
    print("Chua co checkpoint de resume. Hay chay Cell Train Fresh truoc!")

In [ ]:
# Cell Downstream Benchmark (Eval 65 test scans, Linear Probe, Mahalanobis, Sizing Hurdle)
import glob, os
ckpts = sorted(glob.glob(f"{DRIVE_RUNS}/{EXP_NAME}_*/checkpoints/best_model.pt"))
if ckpts:
    best_ckpt = ckpts[-1]
    run_dir = os.path.dirname(os.path.dirname(best_ckpt))
    print(f"Evaluating downstream benchmark on: {best_ckpt}")
    !cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluation.benchmark_cli --checkpoint "{best_ckpt}" --data_dir "$DATA" --output_dir "{run_dir}/evaluation_results"
else:
    print("Chua co best_model.pt de danh gia downstream.")

In [ ]:
# Cell Deep Latent Diagnostics (SVD spectrum decay rate, effective rank R_eff, condition number)
import glob, os
ckpts = sorted(glob.glob(f"{DRIVE_RUNS}/{EXP_NAME}_*/checkpoints/best_model.pt"))
if ckpts:
    best_ckpt = ckpts[-1]
    print(f"Running deep latent diagnostics on: {best_ckpt}")
    !cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.training.diagnostics --checkpoint "{best_ckpt}" --data_dir "$DATA"
else:
    print("Chua co best_model.pt de chay diagnostics.")